# NAT Cluster Analysis — Decision Gate Notebook

**Purpose:** Determine whether natural market regimes exist in the collected feature data.

This notebook runs the full cluster analysis pipeline and answers four questions:
- **Q1:** Do natural clusters exist? (Silhouette > 0.25, GMM k > 1, dip test)
- **Q2:** Are clusters stable? (Bootstrap ARI > 0.6, temporal ARI > 0.5, self-transition > 0.7)
- **Q3:** Do clusters predict returns? (Kruskal-Wallis p < 0.05, eta-squared > 0.01)
- **Q4:** At which timeframe and subspace?

**Decision rules:**
- **GO:** Majority YES on Q1-Q3 → proceed to phase detection
- **PIVOT:** Clusters exist but don't predict returns → try different features/timeframes
- **NO-GO:** No clusters found → accept null result

---

In [ ]:
# Setup — imports and configuration
import sys
import os
import warnings

# Add scripts/ to path so cluster_pipeline is importable
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "scripts"))
# If running from notebooks/ dir:
if not any("cluster_pipeline" in p for p in sys.path):
    sys.path.insert(0, os.path.join(os.getcwd(), "..", "scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cluster_pipeline.config import (
    FEATURE_VECTORS, COMPOSITE_VECTORS, get_vector_columns,
    extract_vector, list_vectors, print_vectors,
)
from cluster_pipeline.loader import (
    load_parquet, scan_schema, print_schema_summary, validate_schema,
)
from cluster_pipeline.preprocess import (
    TIMEFRAMES, aggregate_bars, aggregate_multi_timeframe,
    preprocess, bar_summary,
)
from cluster_pipeline.cluster import (
    fit_gmm, fit_gmm_auto, k_sweep, cluster_quality,
    bootstrap_stability, temporal_stability, dip_test,
    bimodality_coefficient, multimodality_scan, predictive_quality,
    full_analysis, compute_linkage,
)
from cluster_pipeline.reduce import (
    fit_pca, fit_tsne, reduce_all, pca_optimal_components,
    top_pca_features, compare_projections, projection_summary,
)
from cluster_pipeline.viz import (
    plot_scatter_2d, plot_scatter_continuous, plot_comparison_grid,
    plot_silhouette, plot_centroid_heatmap, plot_pairplot,
    plot_dendrogram, plot_k_sweep, generate_all_plots,
)

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline
plt.rcParams["figure.dpi"] = 120

print("All pipeline modules loaded successfully.")

## 1. Configuration

Set the data directory and analysis parameters. Adjust these before running.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# Point DATA_DIR at your parquet output from the Rust ingestor.
# The notebook will auto-detect schema and available vectors.

DATA_DIR = "../data/features"  # relative to notebooks/

# Analysis scope
SYMBOLS = None          # None = all symbols; or e.g. ["BTC", "ETH", "SOL"]
TIMEFRAME = "15min"     # Primary analysis timeframe (5min, 15min, 1h, 4h)
VECTORS_TO_TEST = [     # Feature vectors to cluster independently
    "entropy",
    "trend",
    "volatility",
    "toxicity",
    "regime",
    "micro",            # composite: entropy + volatility + flow
]

# Clustering parameters
K_RANGE = range(2, 11)          # k values to sweep
N_BOOTSTRAP = 50                # bootstrap resamples for stability
SCALER = "zscore"               # zscore, minmax, robust, none
RANDOM_STATE = 42

# Decision gate thresholds (from spec)
SILHOUETTE_THRESHOLD = 0.25
BOOTSTRAP_ARI_THRESHOLD = 0.6
TEMPORAL_ARI_THRESHOLD = 0.5
SELF_TRANSITION_THRESHOLD = 0.7
KRUSKAL_P_THRESHOLD = 0.05
ETA_SQUARED_THRESHOLD = 0.01

print(f"Data directory: {DATA_DIR}")
print(f"Timeframe:      {TIMEFRAME}")
print(f"Vectors:        {VECTORS_TO_TEST}")
print(f"k range:        {list(K_RANGE)}")

## 2. Data Loading & Schema Inspection

Load parquet files and verify the schema matches expected feature vectors.

In [ ]:
# Schema scan (metadata only — no data loaded yet)
schema_info = scan_schema(DATA_DIR)
print_schema_summary(DATA_DIR)

print(f"\n{'='*60}")
print(f"Total files:   {schema_info['file_count']}")
print(f"Total rows:    {schema_info['total_rows']:,}")
print(f"Total columns: {len(schema_info['columns'])}")
print(f"Symbols found: {schema_info['symbols']}")

In [ ]:
# Load data
df = load_parquet(DATA_DIR, symbols=SYMBOLS)

# Validate schema
validation = validate_schema(df, require_meta=True)
print(f"Schema valid:         {validation['valid']}")
print(f"Rows loaded:          {validation['row_count']:,}")
print(f"Vectors available:    {validation['vectors_available']}")
print(f"Vectors complete:     {validation['vectors_complete']}")

if validation['errors']:
    print(f"\nERRORS:")
    for e in validation['errors']:
        print(f"  ✗ {e}")

if validation['warnings']:
    print(f"\nWARNINGS ({len(validation['warnings'])}):")
    for w in validation['warnings'][:10]:
        print(f"  ⚠ {w}")
    if len(validation['warnings']) > 10:
        print(f"  ... and {len(validation['warnings']) - 10} more")

# Check which test vectors are available
available_vectors = set(validation['vectors_available'])
vectors_ok = [v for v in VECTORS_TO_TEST if v in available_vectors or v in ("micro", "macro", "full")]
vectors_missing = [v for v in VECTORS_TO_TEST if v not in vectors_ok]
print(f"\nVectors to analyze: {vectors_ok}")
if vectors_missing:
    print(f"Vectors unavailable: {vectors_missing}")

In [ ]:
# Vector coverage overview
print_vectors(df)

## 3. Bar Aggregation

Convert 100ms tick data into time bars at the configured timeframe.
Category-specific rules: price→OHLC, volume→sum, entropy→mean+slope, default→mean+std+last.

In [ ]:
# Aggregate at primary timeframe
bars = aggregate_bars(df, timeframe=TIMEFRAME)
summary = bar_summary(bars)

print(f"Timeframe:    {TIMEFRAME}")
print(f"Bars:         {summary['n_bars']}")
print(f"Features:     {summary['n_features']}")
print(f"Symbols:      {summary['symbols']}")
print(f"Time range:   {summary['time_range'].get('start', 'N/A')} → {summary['time_range'].get('end', 'N/A')}")

if summary['tick_count_stats']:
    tc = summary['tick_count_stats']
    print(f"Ticks/bar:    min={tc['min']}, median={tc['median']:.0f}, mean={tc['mean']:.0f}, max={tc['max']}")

# NaN overview
nan_cols = summary.get('nan_fraction', {})
if nan_cols:
    print(f"\nColumns with NaN: {len(nan_cols)}")
    worst = sorted(nan_cols.items(), key=lambda x: -x[1])[:5]
    for col, frac in worst:
        print(f"  {col}: {frac:.1%}")
else:
    print("\nNo NaN values in aggregated bars.")

In [ ]:
# Optional: compare multiple timeframes
all_timeframes = aggregate_multi_timeframe(df, timeframes=["5min", "15min", "1h", "4h"])

print(f"{'Timeframe':<10} {'Bars':>6} {'Features':>10}")
print("-" * 30)
for tf, tf_bars in all_timeframes.items():
    tf_summary = bar_summary(tf_bars)
    print(f"{tf:<10} {tf_summary['n_bars']:>6} {tf_summary['n_features']:>10}")

## 4. Full Cluster Analysis — Per Vector

For each feature vector, run the complete pipeline:
1. Preprocess (NaN handling, scaling, outlier clipping)
2. k-sweep (GMM, silhouette, BIC)
3. Best-k clustering
4. Quality metrics
5. Bootstrap + temporal stability
6. Multimodality scan
7. Dimensionality reduction (PCA + t-SNE)
8. All diagnostic plots

Results are collected into `analysis_results` for the decision gate.

In [ ]:
# ── Main analysis loop ─────────────────────────────────────────────────────
analysis_results = {}

for vector_name in vectors_ok:
    print(f"\n{'='*70}")
    print(f"  VECTOR: {vector_name}")
    print(f"{'='*70}")

    # ── 4a. Preprocess ────────────────────────────────────────────────────
    try:
        X, feat_cols, meta = preprocess(
            bars, vector=vector_name, scaler=SCALER, clip_sigma=5.0,
        )
    except ValueError as e:
        print(f"  SKIP: {e}")
        analysis_results[vector_name] = {"status": "skip", "reason": str(e)}
        continue

    print(f"  Shape after preprocess: {X.shape}  ({len(feat_cols)} features, {X.shape[0]} bars)")

    # Need enough bars for clustering
    if X.shape[0] < max(K_RANGE) + 1:
        print(f"  SKIP: only {X.shape[0]} bars, need at least {max(K_RANGE)+1}")
        analysis_results[vector_name] = {"status": "skip", "reason": "too few bars"}
        continue

    # ── 4b. Full analysis (k-sweep, GMM, quality, stability, modality) ──
    print("  Running full_analysis (k-sweep, stability, multimodality)...")
    analysis = full_analysis(
        X,
        k_range=K_RANGE,
        n_bootstrap=N_BOOTSTRAP,
        column_names=feat_cols,
        random_state=RANDOM_STATE,
    )

    sweep = analysis["sweep"]
    best_k = analysis["best_k"]
    best_result = analysis["best_result"]
    quality = analysis["quality"]
    boot = analysis["bootstrap_stability"]
    temp = analysis["temporal_stability"]
    modality = analysis["multimodality"]

    print(f"  Best k (BIC):       {sweep.best_k_bic}")
    print(f"  Best k (Silhouette):{sweep.best_k_silhouette}")
    print(f"  Selected k:         {best_k}")
    print(f"  Silhouette:         {quality.silhouette:.4f}  (threshold: {SILHOUETTE_THRESHOLD})")
    print(f"  Davies-Bouldin:     {quality.davies_bouldin:.4f}  (lower is better)")
    print(f"  Bootstrap ARI:      {boot.mean_ari:.4f} ± {boot.std_ari:.4f}  (threshold: {BOOTSTRAP_ARI_THRESHOLD})")
    print(f"  Temporal ARI:       {temp.mean_ari:.4f} ± {temp.std_ari:.4f}  (threshold: {TEMPORAL_ARI_THRESHOLD})")
    print(f"  Cluster sizes:      {quality.cluster_sizes}")

    # Multimodal features
    n_multimodal = sum(1 for m in modality if m["multimodal"])
    print(f"  Multimodal features:{n_multimodal}/{len(modality)}")
    if modality:
        top3 = modality[:3]
        for m in top3:
            print(f"    {m['feature']}: dip_p={m['dip_p_value']:.4f}, BC={m['bimodality_coefficient']:.4f}")

    # ── 4c. Dimensionality reduction ──────────────────────────────────────
    print("  Running dimensionality reduction...")
    projections = reduce_all(X, n_components=2, random_state=RANDOM_STATE)
    pca_result = projections["pca"]

    # PCA variance explained
    if pca_result.cumulative_variance is not None:
        cum_var = pca_result.cumulative_variance
        print(f"  PCA variance (2D):  {cum_var[-1]:.1%}")
        n_95 = pca_optimal_components(X, variance_threshold=0.95)
        print(f"  PCA dims for 95%:   {n_95}")

    # Top PCA features
    pca_top = top_pca_features(pca_result, feat_cols, n_top=5)
    for pc in pca_top:
        feats = ", ".join(f"{f['feature']}({f['loading']:+.3f})" for f in pc['top_features'][:3])
        print(f"  PC{pc['component']} ({pc['explained_variance_ratio']:.1%}): {feats}")

    # Compare projections
    proj_comparison = compare_projections(list(projections.values()), best_result.labels)
    for key, info in proj_comparison.items():
        print(f"  Silhouette in {key}: {info['silhouette_2d']:.4f}")

    # ── 4d. Visualizations ────────────────────────────────────────────────
    print("  Generating plots...")

    # Compute linkage for dendrogram (subsample if large)
    X_link = X[:500] if len(X) > 500 else X
    linkage_mat = compute_linkage(X_link, method="ward")

    # Use PCA embedding for plots
    embedding_2d = pca_result.embedding

    plots = generate_all_plots(
        X, best_result.labels, embedding_2d,
        feature_names=feat_cols,
        sweep=sweep,
        linkage_matrix=linkage_mat,
        method_name=f"PCA — {vector_name}",
    )

    # Display key plots inline
    for name, fig in plots.items():
        fig.suptitle(f"{vector_name} / {TIMEFRAME} — {name}", fontsize=11)
        plt.show()
        plt.close(fig)

    # t-SNE scatter if available
    if "tsne" in projections:
        fig_tsne = plot_scatter_2d(
            projections["tsne"].embedding, best_result.labels,
            title=f"t-SNE — {vector_name} (k={best_k})",
        )
        plt.show()
        plt.close(fig_tsne)

    # ── 4e. Store results ─────────────────────────────────────────────────
    analysis_results[vector_name] = {
        "status": "ok",
        "X_shape": X.shape,
        "feat_cols": feat_cols,
        "best_k": best_k,
        "silhouette": quality.silhouette,
        "davies_bouldin": quality.davies_bouldin,
        "bootstrap_ari": boot.mean_ari,
        "temporal_ari": temp.mean_ari,
        "n_multimodal": n_multimodal,
        "n_features_total": len(modality),
        "pca_var_2d": float(cum_var[-1]) if pca_result.cumulative_variance is not None else None,
        "analysis": analysis,
        "projections": projections,
        "labels": best_result.labels,
    }

print(f"\n{'='*70}")
print(f"  Analysis complete for {len([r for r in analysis_results.values() if r['status'] == 'ok'])} vectors")
print(f"{'='*70}")

## 5. Predictive Quality (Optional)

If forward returns are available (e.g., from midprice bars), test whether clusters predict returns.
This answers **Q3**: Do clusters predict forward returns?

In [ ]:
# ── Compute forward returns from midprice bars ────────────────────────────
# Look for a midprice column in bars (from OHLC aggregation)
midprice_col = None
for candidate in ["raw_midprice_close", "raw_midprice_mean", "raw_microprice_close"]:
    if candidate in bars.columns:
        midprice_col = candidate
        break

if midprice_col is not None:
    forward_returns = bars[midprice_col].pct_change().shift(-1).values
    # Drop last bar (NaN forward return)
    valid_mask = ~np.isnan(forward_returns)
    print(f"Forward returns computed from '{midprice_col}'")
    print(f"  Valid bars: {valid_mask.sum()}/{len(forward_returns)}")
    print(f"  Mean:  {np.nanmean(forward_returns):.6f}")
    print(f"  Std:   {np.nanstd(forward_returns):.6f}")

    # Run predictive quality for each analyzed vector
    print(f"\n{'Vector':<14} {'K-W p':>10} {'eta²':>8} {'Self-Trans':>11} {'Significant':>12}")
    print("-" * 60)

    for vname, res in analysis_results.items():
        if res["status"] != "ok":
            continue
        labels = res["labels"]
        # Align: bars may have been filtered, use the same indices
        n = min(len(labels), len(forward_returns))
        fr = forward_returns[:n]
        lab = labels[:n]
        # Remove NaN returns
        mask = ~np.isnan(fr)
        if mask.sum() < 10:
            print(f"  {vname:<14} — too few valid returns")
            continue

        pq = predictive_quality(lab[mask], fr[mask])
        res["predictive"] = pq

        sig_mark = "YES" if pq["significant"] else "no"
        print(f"  {vname:<14} {pq['kruskal_wallis_p']:>10.4f} {pq['eta_squared']:>8.4f} "
              f"{pq['self_transition_rate']:>11.4f} {sig_mark:>12}")

        # Per-cluster breakdown
        for cid, stats in sorted(pq["per_cluster"].items()):
            print(f"    Cluster {cid}: n={stats['count']}, "
                  f"mean_ret={stats['mean_return']:+.6f}, sharpe={stats['sharpe']:+.3f}")
else:
    print("No midprice column found in bars — skipping predictive quality.")
    print("Available columns with 'mid' or 'price':")
    price_cols = [c for c in bars.columns if "mid" in c.lower() or "price" in c.lower()]
    print(f"  {price_cols[:10]}")
    forward_returns = None

## 6. Comparison Grid — Best Vector

For the best-performing vector, show the 2x2 comparison grid:
cluster labels, entropy, forward return sign, and symbol.

In [ ]:
# Find the best vector by silhouette score
ok_results = {k: v for k, v in analysis_results.items() if v["status"] == "ok"}
if ok_results:
    best_vector = max(ok_results, key=lambda k: ok_results[k]["silhouette"])
    res = ok_results[best_vector]
    print(f"Best vector by silhouette: {best_vector} (sil={res['silhouette']:.4f}, k={res['best_k']})")

    embedding = res["projections"]["pca"].embedding
    labels = res["labels"]

    # Gather optional data for comparison grid
    entropy_vals = None
    for ent_candidate in ["ent_tick_1m_mean", "ent_tick_5s_mean", "ent_permutation_returns_8_mean"]:
        if ent_candidate in bars.columns:
            entropy_vals = bars[ent_candidate].values[:len(labels)]
            break

    symbol_vals = None
    if "symbol" in bars.columns:
        symbol_vals = bars["symbol"].values[:len(labels)]

    fr_vals = forward_returns[:len(labels)] if forward_returns is not None else None

    fig = plot_comparison_grid(
        embedding,
        labels=labels,
        entropy=entropy_vals,
        forward_returns=fr_vals,
        symbols=symbol_vals,
        title=f"Comparison Grid — {best_vector} / {TIMEFRAME}",
    )
    plt.show()
    plt.close(fig)
else:
    print("No vectors were successfully analyzed.")

## 7. Multi-Timeframe Comparison (Optional)

Run the best vector across all 4 timeframes to see if regime structure is timeframe-dependent.

In [ ]:
# Multi-timeframe analysis for the best vector
if ok_results:
    tf_results = {}
    test_vector = best_vector

    print(f"Multi-timeframe analysis for: {test_vector}")
    print(f"\n{'TF':<8} {'Bars':>6} {'k':>3} {'Silhouette':>11} {'Boot ARI':>10} {'Temp ARI':>10} {'Pass Q1':>8} {'Pass Q2':>8}")
    print("-" * 75)

    for tf_name, tf_bars in all_timeframes.items():
        try:
            X_tf, cols_tf, meta_tf = preprocess(
                tf_bars, vector=test_vector, scaler=SCALER, clip_sigma=5.0,
            )
        except ValueError:
            print(f"  {tf_name:<8} — preprocessing failed")
            continue

        if X_tf.shape[0] < max(K_RANGE) + 1:
            print(f"  {tf_name:<8} {X_tf.shape[0]:>6} — too few bars")
            continue

        analysis_tf = full_analysis(
            X_tf, k_range=K_RANGE, n_bootstrap=N_BOOTSTRAP,
            column_names=cols_tf, random_state=RANDOM_STATE,
        )

        q_tf = analysis_tf["quality"]
        b_tf = analysis_tf["bootstrap_stability"]
        t_tf = analysis_tf["temporal_stability"]

        q1 = q_tf.silhouette > SILHOUETTE_THRESHOLD
        q2 = b_tf.mean_ari > BOOTSTRAP_ARI_THRESHOLD and t_tf.mean_ari > TEMPORAL_ARI_THRESHOLD

        print(f"  {tf_name:<8} {X_tf.shape[0]:>6} {analysis_tf['best_k']:>3} "
              f"{q_tf.silhouette:>11.4f} {b_tf.mean_ari:>10.4f} {t_tf.mean_ari:>10.4f} "
              f"{'YES' if q1 else 'no':>8} {'YES' if q2 else 'no':>8}")

        tf_results[tf_name] = {
            "best_k": analysis_tf["best_k"],
            "silhouette": q_tf.silhouette,
            "bootstrap_ari": b_tf.mean_ari,
            "temporal_ari": t_tf.mean_ari,
            "q1_pass": q1,
            "q2_pass": q2,
        }
else:
    print("No vectors available for multi-timeframe analysis.")

---

## 8. Decision Gate Summary

Answering the four key questions across all vectors and timeframes.

| Question | Metric | Threshold | Rule |
|----------|--------|-----------|------|
| **Q1:** Natural clusters exist? | Silhouette > 0.25, GMM k > 1, dip test | Majority features multimodal | Must pass |
| **Q2:** Clusters stable? | Bootstrap ARI > 0.6, Temporal ARI > 0.5 | Both must pass | Must pass |
| **Q3:** Predict returns? | Kruskal-Wallis p < 0.05, eta² > 0.01 | Both must pass | For GO |
| **Q4:** Which timeframe/subspace? | Best silhouette across TF × vector grid | — | Informational |

**Decision:**
- **GO:** Majority YES on Q1-Q3 → proceed to phase detection
- **PIVOT:** Clusters exist but don't predict returns → try different features/timeframes  
- **NO-GO:** No clusters found → accept null result

In [ ]:
# ── Decision Gate Evaluation ───────────────────────────────────────────────

gate_rows = []

for vname, res in analysis_results.items():
    if res["status"] != "ok":
        gate_rows.append({
            "vector": vname, "status": "SKIP", "q1": "—", "q2": "—", "q3": "—",
            "silhouette": "—", "boot_ari": "—", "temp_ari": "—",
        })
        continue

    # Q1: Do natural clusters exist?
    sil = res["silhouette"]
    k = res["best_k"]
    n_mm = res["n_multimodal"]
    n_feat = res["n_features_total"]
    q1_sil = sil > SILHOUETTE_THRESHOLD
    q1_k = k > 1
    q1_mm = n_mm > n_feat / 2  # majority of features are multimodal
    q1 = q1_sil and q1_k

    # Q2: Are clusters stable?
    b_ari = res["bootstrap_ari"]
    t_ari = res["temporal_ari"]
    q2_boot = b_ari > BOOTSTRAP_ARI_THRESHOLD
    q2_temp = t_ari > TEMPORAL_ARI_THRESHOLD
    q2 = q2_boot and q2_temp

    # Q3: Do clusters predict returns?
    pq = res.get("predictive")
    if pq is not None:
        q3_kw = pq["kruskal_wallis_p"] < KRUSKAL_P_THRESHOLD
        q3_eta = pq["eta_squared"] > ETA_SQUARED_THRESHOLD
        q3_trans = pq["self_transition_rate"] > SELF_TRANSITION_THRESHOLD
        q3 = q3_kw and q3_eta
    else:
        q3_kw = q3_eta = q3_trans = None
        q3 = None

    gate_rows.append({
        "vector": vname,
        "k": k,
        "silhouette": f"{sil:.4f}",
        "boot_ari": f"{b_ari:.4f}",
        "temp_ari": f"{t_ari:.4f}",
        "multimodal": f"{n_mm}/{n_feat}",
        "q1": "YES" if q1 else "no",
        "q2": "YES" if q2 else "no",
        "q3": "YES" if q3 else ("N/A" if q3 is None else "no"),
        "self_trans": f"{pq['self_transition_rate']:.3f}" if pq else "N/A",
    })

# Display as table
gate_df = pd.DataFrame(gate_rows)
print("\n" + "=" * 90)
print("  DECISION GATE — Per-Vector Summary")
print("=" * 90)
print(gate_df.to_string(index=False))

# Overall decision
q1_yes = sum(1 for r in gate_rows if r.get("q1") == "YES")
q2_yes = sum(1 for r in gate_rows if r.get("q2") == "YES")
q3_yes = sum(1 for r in gate_rows if r.get("q3") == "YES")
n_ok = sum(1 for r in gate_rows if r.get("q1") in ("YES", "no"))

print(f"\n{'─'*50}")
print(f"  Q1 (Clusters exist):      {q1_yes}/{n_ok} vectors pass")
print(f"  Q2 (Clusters stable):     {q2_yes}/{n_ok} vectors pass")
print(f"  Q3 (Predict returns):     {q3_yes}/{n_ok} vectors pass")
print(f"{'─'*50}")

if q1_yes > n_ok / 2 and q2_yes > n_ok / 2 and q3_yes > 0:
    decision = "GO"
    reason = "Majority pass Q1-Q2, at least one passes Q3 → proceed to phase detection"
elif q1_yes > n_ok / 2 and q2_yes > 0:
    decision = "PIVOT"
    reason = "Clusters exist but predictive power is weak → try different features or timeframes"
else:
    decision = "NO-GO"
    reason = "No reliable cluster structure found → accept null result"

print(f"\n  DECISION: {decision}")
print(f"  Reason:   {reason}")
print(f"{'='*90}")

---

## Next Steps

Based on the decision above:

- **GO:** Best vector + timeframe identified. Proceed to:
  - Implement online phase detector using the winning GMM
  - Build transition matrix for regime switching
  - Integrate with backtesting framework

- **PIVOT:** Try:
  - Different feature combinations or composite vectors
  - Alternative timeframes (check multi-TF results above)
  - HDBSCAN instead of GMM (density-based, auto-k)
  - Feature engineering: rolling windows, cross-asset features

- **NO-GO:** The data may not contain exploitable regime structure at these timeframes.
  - Consider longer collection periods
  - Try tick-level clustering (no bar aggregation)
  - Re-examine feature engineering in the Rust ingestor